# EdX Dataset Cleaning

This notebook performs data cleaning on the EdX.csv dataset. The cleaning steps include:
1. Reading and examining the data
2. Handling missing values
3. Removing duplicates
4. Standardizing text formats
5. Cleaning special characters
6. Processing Skills/Topics columns
7. Saving the cleaned dataset

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import re

# Read the dataset
df = pd.read_csv('../Data/raw/EdX.csv')

In [2]:
# Examine the data
print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing Values:\n", df.isnull().sum())
print("\nData Types:\n", df.dtypes)
print("\nSample of first few rows:\n", df.head())

Dataset Shape: (720, 6)

Columns: ['Name', 'University', 'Difficulty Level', 'Link', 'About', 'Course Description']

Missing Values:
 Name                  0
University            0
Difficulty Level      0
Link                  0
About                 0
Course Description    0
dtype: int64

Data Types:
 Name                  object
University            object
Difficulty Level      object
Link                  object
About                 object
Course Description    object
dtype: object

Sample of first few rows:
                                                 Name  \
0                                How to Learn Online   
1  Programming for Everybody (Getting Started wit...   
2            CS50's Introduction to Computer Science   
3                                 The Analytics Edge   
4  Marketing Analytics: Marketing Measurement Str...   

                              University Difficulty Level  \
0                                    edX         Beginner   
1             The Un

In [3]:
# 1. Clean special characters and standardize text
def clean_text(text):
    if pd.isna(text):
        return text
    # Replace special characters and standardize text
    text = str(text)
    text = text.replace('�', "'")  # Replace smart quotes
    text = text.replace('�', '-')  # Replace em dash
    text = text.replace('�', 'e')  # Replace accented e
    text = re.sub(r'\s+', ' ', text)  # Remove multiple spaces
    return text.strip()

# Apply cleaning to text columns
text_columns = df.select_dtypes(include=['object']).columns
for col in text_columns:
    df[col] = df[col].apply(clean_text)

# 2. Handle missing values
df = df.replace('', np.nan)  # Convert empty strings to NaN
df = df.replace('N/A', np.nan)  # Convert 'N/A' to NaN

# 3. Convert numeric fields if they exist
numeric_cols = ['Course Rating', 'Enrolled Students']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].str.replace(',', '').str.replace('k', '000'), errors='coerce')

# 4. Clean and standardize URL field if it exists
if 'Course URL' in df.columns:
    df['Course URL'] = df['Course URL'].str.strip()

# 5. Process Topics/Skills column if it exists
def process_topics(topics):
    if pd.isna(topics):
        return topics
    # Split topics by common separators
    topics_list = re.split(r'[,|;]\s*', str(topics))
    # Clean individual topics
    topics_list = [topic.strip() for topic in topics_list]
    # Remove empty topics
    topics_list = [topic for topic in topics_list if topic]
    return ' | '.join(topics_list)

if 'Topics' in df.columns:
    df['Topics'] = df['Topics'].apply(process_topics)
elif 'Skills' in df.columns:
    df['Skills'] = df['Skills'].apply(process_topics)

# 6. Remove duplicates
if 'Course Name' in df.columns and 'University' in df.columns:
    df.drop_duplicates(subset=['Course Name', 'University'], inplace=True)
else:
    df.drop_duplicates(inplace=True)

# 7. Reset index
df.reset_index(drop=True, inplace=True)

In [4]:
# Examine the cleaned data
print("Final Dataset Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
print("\nData Types:\n", df.dtypes)

# Display sample of cleaned data
print("\nSample of cleaned data:\n")
print(df.head())

# Save the cleaned dataset
output_path = '../Data/processed/edx_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"\nCleaned dataset saved to: {output_path}")

Final Dataset Shape: (719, 6)

Missing Values:
 Name                  0
University            0
Difficulty Level      0
Link                  0
About                 0
Course Description    0
dtype: int64

Data Types:
 Name                  object
University            object
Difficulty Level      object
Link                  object
About                 object
Course Description    object
dtype: object

Sample of cleaned data:

                                                Name  \
0                                How to Learn Online   
1  Programming for Everybody (Getting Started wit...   
2            CS50's Introduction to Computer Science   
3                                 The Analytics Edge   
4  Marketing Analytics: Marketing Measurement Str...   

                              University Difficulty Level  \
0                                    edX         Beginner   
1             The University of Michigan         Beginner   
2                     Harvard University       

# Data Cleaning Summary

The following cleaning steps were performed on the EdX dataset:

1. **Special Characters Cleaning**:
   - Replaced special quotes, dashes, and accented characters
   - Removed multiple spaces
   - Standardized text formatting

2. **Missing Values**:
   - Converted empty strings to NaN
   - Converted 'N/A' to NaN
   - Handled missing values in numeric fields

3. **Numeric Data Processing**:
   - Converted numeric fields to proper numeric types
   - Handled 'k' suffixes in enrollment numbers
   - Removed commas from numeric values

4. **URL Standardization**:
   - Cleaned and standardized course URLs
   - Removed leading/trailing whitespace

5. **Topics/Skills Processing**:
   - Split topics by common separators (comma, pipe, semicolon)
   - Cleaned individual topics
   - Joined with standard separator ' | '

6. **Data Quality**:
   - Removed duplicate entries
   - Reset index for clean data structure

The cleaned dataset has been saved to: '../Data/processed/edx_cleaned.csv'